In [1]:
# ============================================
# 02_limpieza.ipynb — Limpieza y normalización
# Proyecto: DataZ Movility (Fase A — Bizi)
# Autor: Miguel
# ============================================

import pandas as pd
import geopandas as gpd
from pathlib import Path

# Rutas del proyecto
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

print("Entorno cargado correctamente.")


Entorno cargado correctamente.


In [2]:
bizi_path = DATA_RAW / "bizi.geojson"

try:
    gdf = gpd.read_file(bizi_path)
    print("Dataset cargado correctamente.")
except Exception as e:
    print("Error al cargar el dataset:", e)

gdf.head()


Dataset cargado correctamente.


,id,about,title,estado,estadoEstacion,address,tipoEquipamiento,bicisDisponibles,anclajesDisponibles,lastUpdated,description,descripcion,icon,geometry
0,279,http://www.zaragoza.es/ciudad/viapublica/movil...,193- Pza. La Ermita,IN_SERVICE,http://vocab.linkeddata.es/datosabiertos/kos/t...,193- Pza. La Ermita,http://vocab.linkeddata.es/datosabiertos/kos/u...,1,18,2026-02-25 17:08:07+00:00,<ul><li>Estado: Operativa</li><li>Bicis dispon...,Datos de la estación bizi 193- Pza. La Ermita,//www.zaragoza.es/contenidos/iconos/bizi/conbi...,POINT (-0.91122 41.6315)
1,278,http://www.zaragoza.es/ciudad/viapublica/movil...,270- Delicias Autobuses,IN_SERVICE,http://vocab.linkeddata.es/datosabiertos/kos/t...,270- Delicias Autobuses,http://vocab.linkeddata.es/datosabiertos/kos/u...,0,19,2026-02-25 17:08:07+00:00,<ul><li>Estado: Operativa</li><li>Bicis dispon...,Datos de la estación bizi 270- Delicias Autobuses,//www.zaragoza.es/contenidos/iconos/bizi/sinbi...,POINT (-0.91078 41.65952)
2,277,http://www.zaragoza.es/ciudad/viapublica/movil...,150- Mrio. Siresa: Dr. Iranzo,IN_SERVICE,http://vocab.linkeddata.es/datosabiertos/kos/t...,150- Mrio. Siresa: Dr. Iranzo,http://vocab.linkeddata.es/datosabiertos/kos/u...,11,8,2026-02-25 17:08:07+00:00,<ul><li>Estado: Operativa</li><li>Bicis dispon...,Datos de la estación bizi 150- Mrio. Siresa: D...,//www.zaragoza.es/contenidos/iconos/bizi/conbi...,POINT (-0.8641 41.64759)
3,274,http://www.zaragoza.es/ciudad/viapublica/movil...,190- Puerto Venecia Este,IN_SERVICE,http://vocab.linkeddata.es/datosabiertos/kos/t...,190- Puerto Venecia Este,http://vocab.linkeddata.es/datosabiertos/kos/u...,26,15,2026-02-25 17:08:07+00:00,<ul><li>Estado: Operativa</li><li>Bicis dispon...,Datos de la estación bizi 190- Puerto Venecia ...,//www.zaragoza.es/contenidos/iconos/bizi/conbi...,POINT (-0.88395 41.60822)
4,273,http://www.zaragoza.es/ciudad/viapublica/movil...,224- Manuel Viola,IN_SERVICE,http://vocab.linkeddata.es/datosabiertos/kos/t...,224- Manuel Viola,http://vocab.linkeddata.es/datosabiertos/kos/u...,3,16,2026-02-25 17:08:07+00:00,<ul><li>Estado: Operativa</li><li>Bicis dispon...,Datos de la estación bizi 224- Manuel Viola,//www.zaragoza.es/contenidos/iconos/bizi/conbi...,POINT (-0.8573 41.65914)


In [3]:
# Convertir lastUpdated a datetime
gdf["lastUpdated"] = pd.to_datetime(gdf["lastUpdated"], errors="coerce")

# Asegurar tipos numéricos
gdf["bicisDisponibles"] = pd.to_numeric(gdf["bicisDisponibles"], errors="coerce")
gdf["anclajesDisponibles"] = pd.to_numeric(gdf["anclajesDisponibles"], errors="coerce")

# Convertir estado a categoría
gdf["estado"] = gdf["estado"].astype("category")

gdf.dtypes


id                                     str
about                                  str
title                                  str
estado                            category
estadoEstacion                         str
address                                str
tipoEquipamiento                       str
bicisDisponibles                     int32
anclajesDisponibles                  int32
lastUpdated            datetime64[ms, UTC]
description                            str
descripcion                            str
icon                                   str
geometry                          geometry
dtype: object

In [4]:
gdf = gdf.rename(columns={
    "id": "station_id",
    "title": "station_name",
    "bicisDisponibles": "bikes",
    "anclajesDisponibles": "slots",
    "estado": "status"
})

gdf[["station_id", "station_name", "bikes", "slots", "status"]].head()


,station_id,station_name,bikes,slots,status
0,279,193- Pza. La Ermita,1,18,IN_SERVICE
1,278,270- Delicias Autobuses,0,19,IN_SERVICE
2,277,150- Mrio. Siresa: Dr. Iranzo,11,8,IN_SERVICE
3,274,190- Puerto Venecia Este,26,15,IN_SERVICE
4,273,224- Manuel Viola,3,16,IN_SERVICE


In [5]:
# Asegurar CRS correcto
if gdf.crs is None:
    gdf = gdf.set_crs(epsg=4326)

# Extraer coordenadas
gdf["lon"] = gdf.geometry.x
gdf["lat"] = gdf.geometry.y

gdf[["station_id", "lon", "lat"]].head()


,station_id,lon,lat
0,279,-0.911216,41.631495
1,278,-0.910784,41.659517
2,277,-0.864099,41.647586
3,274,-0.883953,41.608225
4,273,-0.857295,41.659139


In [6]:
import re

def clean_html(text):
    if pd.isna(text):
        return text
    return re.sub("<.*?>", "", text)

gdf["description_clean"] = gdf["description"].apply(clean_html)

gdf[["description", "description_clean"]].head()


,description,description_clean
0,<ul><li>Estado: Operativa</li><li>Bicis dispon...,Estado: OperativaBicis disponibles: 1Anclajes ...
1,<ul><li>Estado: Operativa</li><li>Bicis dispon...,Estado: OperativaBicis disponibles: 0Anclajes ...
2,<ul><li>Estado: Operativa</li><li>Bicis dispon...,Estado: OperativaBicis disponibles: 11Anclajes...
3,<ul><li>Estado: Operativa</li><li>Bicis dispon...,Estado: OperativaBicis disponibles: 26Anclajes...
4,<ul><li>Estado: Operativa</li><li>Bicis dispon...,Estado: OperativaBicis disponibles: 3Anclajes ...


In [7]:
# Capacidad total
gdf["capacity"] = gdf["bikes"] + gdf["slots"]

# Ratio de ocupación
gdf["ratio_ocupacion"] = gdf["bikes"] / gdf["capacity"]

# Categoría de ocupación
def categorize_ratio(r):
    if pd.isna(r):
        return "desconocido"
    if r >= 0.7:
        return "alta"
    if r >= 0.3:
        return "media"
    return "baja"

gdf["ocupacion_categoria"] = gdf["ratio_ocupacion"].apply(categorize_ratio)

gdf[[
    "station_id", "bikes", "slots", "capacity",
    "ratio_ocupacion", "ocupacion_categoria"
]].head()


,station_id,bikes,slots,capacity,ratio_ocupacion,ocupacion_categoria
0,279,1,18,19,0.052632,baja
1,278,0,19,19,0.000000,baja
2,277,11,8,19,0.578947,media
3,274,26,15,41,0.634146,media
4,273,3,16,19,0.157895,baja


In [8]:
anomalos = gdf[
    (gdf["bikes"] < 0) |
    (gdf["slots"] < 0) |
    (gdf["capacity"] <= 0)
]

anomalos


,station_id,about,station_name,status,estadoEstacion,address,tipoEquipamiento,bikes,slots,lastUpdated,description,descripcion,icon,geometry,lon,lat,description_clean,capacity,ratio_ocupacion,ocupacion_categoria


In [9]:
cols_finales = [
    "station_id",
    "station_name",
    "status",
    "bikes",
    "slots",
    "capacity",
    "ratio_ocupacion",
    "ocupacion_categoria",
    "lon",
    "lat",
    "lastUpdated"
]

df_clean = gdf[cols_finales].copy()
df_clean.head()


,station_id,station_name,status,bikes,slots,capacity,ratio_ocupacion,ocupacion_categoria,lon,lat,lastUpdated
0,279,193- Pza. La Ermita,IN_SERVICE,1,18,19,0.052632,baja,-0.911216,41.631495,2026-02-25 17:08:07+00:00
1,278,270- Delicias Autobuses,IN_SERVICE,0,19,19,0.000000,baja,-0.910784,41.659517,2026-02-25 17:08:07+00:00
2,277,150- Mrio. Siresa: Dr. Iranzo,IN_SERVICE,11,8,19,0.578947,media,-0.864099,41.647586,2026-02-25 17:08:07+00:00
3,274,190- Puerto Venecia Este,IN_SERVICE,26,15,41,0.634146,media,-0.883953,41.608225,2026-02-25 17:08:07+00:00
4,273,224- Manuel Viola,IN_SERVICE,3,16,19,0.157895,baja,-0.857295,41.659139,2026-02-25 17:08:07+00:00


In [10]:
output_path = DATA_PROCESSED / "bizi_clean.csv"
df_clean.to_csv(output_path, index=False, encoding="utf-8")

print("Dataset limpio guardado en:", output_path)


Dataset limpio guardado en: ..\data\processed\bizi_clean.csv
